```bash
docker run -v D:\:/data  -v C:\Users\ferat\Documents\GitHub\meeg-pipelines:/meeg-pipelines --env FS_LICENSE="/license.txt"  -p 8080:8080  ghcr.io/vferat/meegpype:dev jupyter notebook --ip=0.0.0.0 --port=8080 --allow-root --notebook-dir=/data
```

```bash
docker run -v D:\:/data  -v C:\Users\ferat\Documents\GitHub\meg_pipelines\license.txt:/license.txt --env FS_LICENSE="/license.txt" freesurfer/freesurfer:7.2.0
```

In [ ]:
%env FS_LICENSE=/meeg-pipelines/license.txt

In [2]:
from meegpype.workflows.anat.bem import generate_bem_workflow


subject = 'sub-ENG008'
subject_dir = "/data/proposal/derivatives/meegpype/sourcedata/freesurfer"

wf = generate_bem_workflow()
wf.base_dir = "/data/test_watersheds/workdir/test_watershed_wf"

wf.inputs.inputnode.subject_id = subject
wf.inputs.inputnode.subjects_dir = subject_dir

wf.run('MultiProc', plugin_args={'n_procs': 4})


251105-14:25:00,58 nipype.workflow INFO:
	 Workflow freesurfer_bem_surfaces settings: ['check', 'execution', 'logging', 'monitoring']
251105-14:25:00,272 nipype.workflow INFO:
	 Running in parallel.
251105-14:25:00,273 nipype.workflow INFO:
	 [MultiProc] Running 0 tasks, and 3 jobs ready. Free memory (GB): 56.38/56.38, Free processors: 4/4.
251105-14:25:00,386 nipype.workflow INFO:
	 [Node] Setting-up "freesurfer_bem_surfaces.fs_source" in "/data/test_watersheds/workdir/test_watershed_wf/freesurfer_bem_surfaces/fs_source".
251105-14:25:00,394 nipype.workflow INFO:
	 [Job 1] Cached (freesurfer_bem_surfaces.watershed).
251105-14:25:00,402 nipype.workflow INFO:
	 [Job 2] Cached (freesurfer_bem_surfaces.generate_bem_path).
251105-14:25:00,458 nipype.workflow INFO:
	 [Node] Executing "fs_source" <nipype.interfaces.io.FreeSurferSource>
251105-14:25:00,966 nipype.workflow INFO:
	 [Node] Finished "fs_source", elapsed time 0.503265s.
251105-14:25:02,273 nipype.workflow INFO:
	 [Job 0] Completed

In [ ]:
import mne

subject = 'sub-ENG008'
subject_dir = "/data/proposal/derivatives/meegpype/sourcedata/freesurfer"
surfaces = mne.make_bem_model(subject=subject, subjects_dir=subject_dir, conductivity=(0.3, 0.006, 0.3), ico=4)

bem = mne.make_bem_solution(surfaces)

In [ ]:
import mne

inner_surf = r"/data/test_watersheds/workdir/test_watershed_wf/freesurfer_bem_surfaces/mris_expand_inner_skull/inner_skull.surf"
inner_surf = mne.read_surface(inner_surf,  read_metadata=True)

inner_decim = mne.decimate_surface(inner_surf[0], inner_surf[1], n_triangles=5120, method='sphere')

mne.write_surface('inner_skull.obj', inner_decim[0], inner_decim[1], overwrite=True)
mne.write_surface('inner_skull.surf', inner_decim[0], inner_decim[1], volume_info=inner_surf[2], overwrite=True)

In [ ]:
import mne

subject = 'sub-ENG008'
subject_dir = "/data/proposal/derivatives/meegpype/sourcedata/freesurfer"
surfaces = mne.make_bem_model(subject=subject, subjects_dir=subject_dir, conductivity=(0.3, 0.006, 0.3), ico=4)

bem = mne.make_bem_solution(surfaces)

In [ ]:
import mne

mc = r"/data/test_watersheds/workdir/test_watershed_wf/freesurfer_bem_surfaces/mri_mc/brain_mc.surf"
mc = mne.read_surface(mc)

mne.write_surface('mc_surface.obj', mc[0], mc[1], overwrite=True)

In [ ]:
import mne

outer_skull = r"/data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/bem/outer_skull.surf"
outer_surf = mne.read_surface(outer_skull)

mne.write_surface('outer_surface.obj', outer_surf[0], outer_surf[1], overwrite=True)

In [ ]:
import mne

brain_surf = r"/data/test_watersheds/workdir/test_watershed_wf/freesurfer_bem_surfaces/mris_expand_brain/brain.surf"
brain_mesh = mne.read_surface(brain_surf)

brain_decim = mne.decimate_surface(brain_mesh[0], brain_mesh[1], n_triangles=5120, method='sphere')

# export .obj
mne.write_surface('brain_surface.obj', brain_decim[0], brain_decim[1], overwrite=True)
mne.write_surface('brain_surface.surf', brain_decim[0], brain_decim[1], overwrite=True)

In [ ]:
mne.write_surface('brain_surface.obj', brain_decim[0], brain_decim[1], overwrite=True)

In [ ]:
import subprocess
import nilearn.image

brainmask = fr"/data/proposal/derivatives/meegpype/workdir/meegpype_forward/sub_ENG008_meg_wf/anat_preproc_wf/anat_fit_wf/ds_t1w_mask_wf/ds_anat_mask/smriprep/smriprep/sub-ENG008/anat/sub-ENG008_desc-brain_mask.nii.gz"
T1 = fr"/data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/mri/T1.mgz"

import nilearn
brain = "brain.nii.gz"
mask = nilearn.image.resample_to_img(brainmask, T1, interpolation='nearest')
mask.to_filename("mask.nii.gz")
nilearn.image.math_img("img1 * img2", img1=nilearn.image.load_img(T1), img2=mask).to_filename(brain)

In [ ]:
import os
from meegpype.workflows.anat.fs_bem import generate_fs_bem_workflow


wf = generate_fs_bem_workflow(name='test_watershed_wf')
wf.base_dir = "/data/test_watersheds/workdir"

wf.inputs.inputnode.subject_id = 'sub-ENG008'
wf.inputs.inputnode.subjects_dir = '/data/proposal/derivatives/meegpype/sourcedata/freesurfer'
wf.inputs.inputnode.brainmask = os.path.join(os.getcwd(), "mask.nii.gz")

wf.run('MultiProc', plugin_args={'n_procs': 4})


In [ ]:
# MNE has its own BEM surface fixing
import mne

# This might work better than pymeshfix for BEM surfaces
surfaces = mne.make_bem_model(
    subject='sub-ENG008',
    subjects_dir='/data/proposal/derivatives/meegpype/sourcedata/freesurfer',
    ico=4,  # Decimation level
    conductivity=[0.3, 0.006, 0.3],  # Brain, skull, scalp
    verbose=True
)

In [ ]:
import nibabel as nib


mri_0 = nib.load("/data/test_watersheds/workdir/test_watershed_wf/apply_mask/brain.nii.gz")
mri_1 = nib.load(brain)

# plot differences
import numpy as np

diff = np.abs(mri_0.get_fdata() - mri_1.get_fdata())
diff.sum()

In [ ]:
from meegpype.interfaces.freesurfer import MRIWatershed

mri_watershed = MRIWatershed()
mri_watershed.inputs.n = True
mri_watershed.inputs.invol = brain
mri_watershed.inputs.outvol = "/data/test_bem/brain_watershed.mgz"
mri_watershed.inputs.surf = 'surfaces'
mri_watershed.inputs.useSRAS = True

print(mri_watershed.cmdline)
mri_watershed.run()


In [ ]:
from meegpype.interfaces.freesurfer import MRIWatershed

mri_watershed = MRIWatershed()
mri_watershed.inputs.n = True
mri_watershed.inputs.invol =  "/data/test_watersheds/workdir/test_watershed_wf/apply_mask/brain.nii.gz"
mri_watershed.inputs.outvol = "/data/test_bem/brain_watershed.mgz"
mri_watershed.inputs.surf = 'surfaces'
mri_watershed.inputs.useSRAS = True

print(mri_watershed.cmdline)
mri_watershed.run()


In [ ]:
from meegpype.workflows.anat.fs_bem import generate_fs_bem_workflow
from nipype import Node
from nipype.interfaces.io import DataSink


brainmask = fr"/data/proposal/derivatives/meegpype/workdir/meegpype_forward/sub_ENG008_meg_wf/anat_preproc_wf/anat_fit_wf/ds_t1w_mask_wf/ds_anat_mask/smriprep/smriprep/sub-ENG008/anat/sub-ENG008_desc-brain_mask.nii.gz"


wf = generate_fs_bem_workflow()
wf.base_dir = "/data/test_bem/workdir"

wf.inputs.inputnode.brainmask = brainmask

datasink =  Node(DataSink(base_directory="/data/test_bem"), name="datasink")

wf.connect(wf.get_node('outputnode'), 'brain_surf', datasink, 'surface.@brain_surf')
wf.connect(wf.get_node('outputnode'), 'inner_skull_surf', datasink, 'surface.@inner_skull_surf')
wf.connect(wf.get_node('outputnode'), 'brain_mc', datasink, 'surface.@brain_mc')
wf.connect(wf.get_node('outputnode'), 'brain_mc_smooth', datasink, 'surface.@brain_mc_smooth')

r = wf.run()

In [ ]:
import subprocess
import nilearn.image

brainmask = fr"/data/proposal/derivatives/meegpype/workdir/meegpype_forward/sub_ENG008_meg_wf/anat_preproc_wf/anat_fit_wf/ds_t1w_mask_wf/ds_anat_mask/smriprep/smriprep/sub-ENG008/anat/sub-ENG008_desc-brain_mask.nii.gz"
T1 = fr"/data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/mri/T1.mgz"

import nilearn
brain = "brain.nii.gz"
mask = nilearn.image.resample_to_img(brainmask, T1, interpolation='nearest')
nilearn.image.math_img("img1 * img2", img1=nilearn.image.load_img(T1), img2=mask).to_filename(brain)

cmd = f"mri_watershed -useSRAS -n -surf test {brain} test.mgz"

#subprocess.run(cmd, shell=True, check=True)

In [ ]:
import shutil
import mne
import pymeshfix as mf


surf = mne.read_surface("/data/test_bem/surface/inner_skull.surf", read_metadata=True)
surf_decim = mne.decimate_surface(surf[0], surf[1], n_triangles=5120, method='sphere')



mne.write_surface("/data/test_bem/surface/inner_skull_5k.surf",surf_decim[0], surf_decim[1], volume_info=surf[2], file_format='freesurfer', overwrite=True)


shutil.copy("/data/test_bem/surface/inner_skull_5k.surf", "/data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/bem/inner_skull.surf")

In [ ]:
import mne
import numpy as np

# Load your original surfaces
outer_skull = mne.read_surface("/data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/bem/outer_skull.surf")
inner_skull = mne.read_surface("/data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/bem/inner_skull.surf")

outer_verts, outer_tris = outer_skull
inner_verts, inner_tris = inner_skull

# Use MNE's surface completion/fixing function
from mne.surface import complete_surface_info

# Create surface dict in MNE format
outer_surf = {
    'rr': outer_verts,
    'tris': outer_tris,
    'np': len(outer_verts),
    'ntri': len(outer_tris)
}

# This fixes topology while preserving structure
outer_surf = complete_surface_info(outer_surf, copy=True, do_neighbor_tris=True)

# Or try MNE's specific BEM fixing
from mne.bem import _assert_complete_surface
outer_surf = _assert_complete_surface(outer_surf)

# Test the fixed surface
outer_fixed = {'rr': outer_surf['rr'], 'tris': outer_surf['tris'], 'id': 4}
inner = {'rr': inner_verts, 'tris': inner_tris, 'id': 3}

tot_angle = _get_solids(outer_fixed['rr'][outer_fixed['tris']], inner['rr'])
normalized = tot_angle / (2 * np.pi)
print(f"After MNE fixing:")
print(f"  Good points: {(np.abs(normalized - 1.0) < 1e-5).sum()} / {len(inner['rr'])}")
print(f"  Mean angle: {normalized.mean():.3f}")

In [ ]:
import mne

outer_skull = mne.read_surface("/data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/bem/outer_skull.surf")
inner_skull = mne.read_surface("/data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/bem/inner_skull.surf")

In [ ]:
import mne

subject = 'sub-ENG008'
subject_dir = "/data/proposal/derivatives/meegpype/sourcedata/freesurfer"
surfaces = mne.make_bem_model(subject=subject, subjects_dir=subject_dir, conductivity=(0.3, 0.006, 0.3), ico=None)

bem = mne.make_bem_solution(surfaces)

In [ ]:
! cp /tmp/tmplx48cjrm/freesurfer_bem_surfaces/mris_smooth/brain_smooth.surf	/data/brain_smooth.surf	

In [ ]:
! head /tmp/tmplx48cjrm/freesurfer_bem_surfaces/mris_smooth/command.txt

In [ ]:
! mris_smooth -a 200 -n 200 /tmp/tmps712t6en/freesurfer_bem_surfaces/mri_mc/brain.surf brain_smooth.surf


In [ ]:
import mne

subject = 'sub-ENG008'
subject_dir = "/data/proposal/derivatives/meegpype/sourcedata/freesurfer"
surfaces = mne.make_bem_model(subject=subject, subjects_dir=subject_dir, conductivity=(0.3, 0.006, 0.3))

bem = mne.make_bem_solution(surfaces)




In [ ]:
mne.viz.plot_bem(subject, subjects_dir=subject_dir)

In [ ]:
! ls ./freesurfer_bem_surfaces

In [ ]:
! lta_convert --invol /data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/mri/orig/001.mgz --outvol /data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/mri/orig.mgz --outlta from_T1w_to_fsnative.lta

In [ ]:
import os

subject = "sub-ENG008"
subjects_dir = "/data/proposal/derivatives/meegpype/sourcedata/freesurfer"

brain = fr"data/proposal/derivatives/meegpype/workdir/meegpype_forward/sub_ENG008_meg_wf/anat_preproc_wf/ds_anat_volumes_wf/mask_anat/sub-ENG008_desc-preproc_T1w_masked.nii.gz"
brain_mask = fr"data/proposal/derivatives/meegpype/workdir/meegpype_forward/sub_ENG008_meg_wf/anat_preproc_wf/anat_fit_wf/ds_t1w_mask_wf/ds_anat_mask/smriprep/smriprep/sub-ENG008/anat/sub-ENG008_desc-brain_mask.nii.gz"



mne.bem.make_watershed_bem(subject, subjects_dir=subjects_dir, brainmask='test.mgz', overwrite=True, atlas=True, preflood=10)

In [ ]:
! mri_watershed -atlas -w -useSRAS -surf /data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/bem/watershed/sub-ENG008 /data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/mri/T1.mgz /data/proposal/derivatives/meegpype/sourcedata/freesurfer/sub-ENG008/bem/watershed/test.mgz

In [ ]:
! mri_convert "/data/proposal/derivatives/meegpype/workdir/meegpype_forward/sub_ENG008_meg_wf/anat_preproc_wf/anat_fit_wf/ds_t1w_mask_wf/ds_anat_mask/smriprep/smriprep/sub-ENG008/anat/sub-ENG008_desc-brain_mask.nii.gz" brain.mgz

In [ ]:
! mri_binarize

In [ ]:
! mri_binarize --i brain.mgz --match 0 --inv --o brainmask.mgz

In [ ]:
! mri_tessellate brainmask.mgz 1 brain

In [ ]:
! mris_smooth brain brain_smooth -a 25 -n 25

In [ ]:
! mris_inflate brain_smooth brain_inflated

In [ ]:
! mri_mc 

In [ ]:
! mri_mc brainmask.mgz 1 brainmask_mc

In [ ]:
! mris_smooth -a 200 -n 200 brainmask_mc  brainmask_mc_smooth

In [ ]:
! mris_smooth

In [ ]:
! mris_expand brainmask_mc_smooth 2 brainmask_mc_smooth_expanded

In [ ]:
! mris_convert brainmask_mc_smooth brainmask_mc_smooth.surf.gii

In [ ]:
! wb_command -surface-generate-inflated brainmask_mc_smooth.surf.gii wb_inflated.surf.gii wb_very_inflated.surf.gii 

In [ ]:
! mris_convert wb_inflated.surf.gii wb_inflated.surf